In [1]:
import pandas as pd
df= pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.sample(5)

,question,answer
81,How many colors are in a rainbow?,7
11,Who developed the theory of relativity?,Albert-Einstein
62,What is the opposite of 'day'?,Night
4,What is the boiling point of water in Celsius?,100
59,What is the study of living organisms called?,Biology


In [3]:
def tokenize(text):
  text = text.lower()
  text=text.replace("'"," ")
  text=text.replace("?"," ")
  return text.split()


In [4]:
tokenize("How many colors are in a rainbow?	")

['how', 'many', 'colors', 'are', 'in', 'a', 'rainbow']

In [5]:
vocab ={'<UNK>':0}

In [12]:
def build_vocab(row):
  tokenized_question =tokenize(row['question'])
  tokenized_answer =tokenize(row['answer'])
  merged_tokens = tokenized_question+tokenized_answer
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)


In [13]:
df.apply(build_vocab, axis = 1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [15]:
len(vocab)

324

In [16]:
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text


In [19]:
text_to_indices('What is my name?', vocab)

[1, 2, 0, 0]

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

In [25]:
class qadataset(Dataset):
  def __init__(self,df, vocab):
    self.df =df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [26]:
dataset = qadataset(df, vocab)

In [27]:
data_loader = DataLoader(dataset,batch_size=1, shuffle=True)

In [28]:
for question, answer in data_loader:
  print(question, answer)

tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([[36]])
tensor([[ 1,  2,  3, 24, 25,  5, 26, 19, 27]]) tensor([[28]])
tensor([[ 42, 137, 118,   3, 247,   5, 248]]) tensor([[249]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[  1,   2,   3, 141, 117,  83,   3, 277, 278]]) tensor([[121]])
tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]]) tensor([[273]])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([[259]])
tensor([[ 10,  11, 157, 158, 159]]) tensor([[160]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([[106]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([[156]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[  1,  87, 229, 230, 231, 232]]) tensor([[233]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[ 10,  96,   3, 104, 239]]) tenso

In [29]:
import torch.nn as nn

In [38]:
class rnn(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, 50)
    self.rn = nn.RNN(50, 64, batch_first=True)
    self.output = nn.Linear(64, vocab_size)

  def forward(self,question):
    embeddings = self.embed(question)
    hidden, final  = self.rn(embeddings)
    outputs = self.output(final.squeeze(0))
    return outputs

In [39]:
learning_rate = 0.001
epochs =20


In [40]:
model = rnn(len(vocab))

In [41]:
criterion =nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [43]:
for epoch in range(epochs):
  total_loss = 0
  for question , answer in data_loader:
    prediction = model(question)
    loss = criterion(prediction,answer[0])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  print(f'Epoch:{epoch+1}, Loss:{total_loss}')

Epoch:1, Loss:527.2247543334961
Epoch:2, Loss:459.3116753101349
Epoch:3, Loss:379.91630268096924
Epoch:4, Loss:319.0470623970032
Epoch:5, Loss:265.1887528896332
Epoch:6, Loss:216.6915558576584
Epoch:7, Loss:171.33975911140442
Epoch:8, Loss:132.12414062023163
Epoch:9, Loss:100.30639708042145
Epoch:10, Loss:76.81449216604233
Epoch:11, Loss:59.10617871582508
Epoch:12, Loss:46.04430937767029
Epoch:13, Loss:36.93438324332237
Epoch:14, Loss:30.34458240121603
Epoch:15, Loss:25.15617661178112
Epoch:16, Loss:21.091193914413452
Epoch:17, Loss:18.11950796842575
Epoch:18, Loss:15.494210563600063
Epoch:19, Loss:13.34956495091319
Epoch:20, Loss:11.587797343730927


In [46]:
def predict(model, question , threshold=0.5):
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output=model(question_tensor)
  probs = torch.nn.functional.softmax(output, dim=1)
  value, index = torch.max(probs, axis=1)
  if value<threshold:
    print('Dont know')
  else:
    print(list(vocab.keys())[index])


In [48]:
predict(model,'Who developed the theory of relativity?')

albert-einstein


In [49]:
predict(model,'What is capital of France?')

paris


In [50]:
predict(model, 'What is my name?')

Dont know
